In [3]:
import pandas as pd
from pycaret.regression import *

# 1. Load data from CSV
# Parse the 'timestamp' column as datetime for potential time-aware features
df = pd.read_csv('hourly_cp_summary.csv', parse_dates=['timestamp'])

# 2. Rename columns for standardization and PyCaret compatibility
df = df.rename(columns={
    'timestamp': 'hour_start',                    # Timestamp of the hour group
    'average_vol': 'normalized_volatility',       # Volatility normalized within currency pair
    'average_fd': 'normalized_fd',                # Normalized Fractal Dimension (FD)
    'correlation_with_btc': 'corr'                # BTC correlation as a feature
})

# 3. Filter out rows with missing or zero volatility or FD
# Some pairs (e.g., USDCNY) may not move or lack feature data — exclude them
df = df[(df['normalized_volatility'] > 0) & (df['normalized_fd'] > 0)]


In [4]:
# 4. Define training and testing currency pairs
# These groups are typically based on importance, stability, or data quality
base_cps = ['USDEUR', 'USDCAD', 'USDGBP', 'USDCHF', 'USDAUD']       # Base pairs used for training
remaining_cps = ['EURCHF', 'GBPEUR', 'GBPCHF', 'USDJPY', 'USDINR', 'USDCNY']  # For out-of-sample testing

# 5. Split function to separate training and testing datasets based on currency pair
def split_train_test(df, train_cps, test_cps):
    train_df = df[df['currency_pair'].isin(train_cps)]  # Subset for training
    test_df = df[df['currency_pair'].isin(test_cps)]    # Subset for testing
    return train_df.copy(), test_df.copy()              # Return copies to avoid warnings or leakage

# Execute the split
train_df, test_df = split_train_test(df, base_cps, remaining_cps)


In [5]:
# 6. Train a regression model to predict normalized_volatility

# === Initialize PyCaret setup ===
vol_exp = setup(
    data=train_df,                            # Use only the base currency pairs
    target='normalized_volatility',          # Target variable to predict
    session_id=42,                           # Random seed for reproducibility
    verbose=False,                           # Suppress detailed logs
    ignore_features=['currency_pair', 'hour_start', 'normalized_fd']  # Exclude non-predictive features
)

# === Automatically compare and select the best model (cross-validated on 2 folds) ===
vol_model = compare_models(fold=2)

# === Tune the selected model to optimize its performance ===
vol_model_tuned = tune_model(vol_model, fold=2)

# === Prepare test data by dropping actual target (and unused 'normalized_fd') ===
test_df_predict = test_df.drop(columns=['normalized_volatility', 'normalized_fd'])

# === Predict on the test set using the tuned model ===
# The result includes the prediction in a column named 'prediction_label'
test_df['predicted_vol'] = predict_model(vol_model_tuned, data=test_df_predict)['prediction_label']


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lasso,Lasso Regression,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851,1.5600
lightgbm,Light Gradient Boosting Machine,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851,0.0350
en,Elastic Net,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851,3.0500
dummy,Dummy Regressor,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851,0.0350
omp,Orthogonal Matching Pursuit,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851,0.0250
llar,Lasso Least Angle Regression,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851,2.1250
ada,AdaBoost Regressor,0.0002,0.0000,0.0003,-0.3622,0.0003,0.3516,0.0250
gbr,Gradient Boosting Regressor,0.0002,0.0000,0.0003,-0.6174,0.0003,0.3435,0.0500
rf,Random Forest Regressor,0.0002,0.0000,0.0003,-0.6439,0.0003,0.3743,0.1350
dt,Decision Tree Regressor,0.0002,0.0000,0.0003,-0.6544,0.0003,0.3484,0.0200


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0002,0.0000,0.0003,-0.0949,0.0003,0.3165
1,0.0002,0.0000,0.0002,-0.3862,0.0002,0.4538
Mean,0.0002,0.0000,0.0002,-0.2406,0.0002,0.3851
Std,0.0000,0.0000,0.0001,0.1456,0.0001,0.0686


Fitting 2 folds for each of 10 candidates, totalling 20 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [6]:
# 7. Train a regression model to predict normalized_fd

# === Initialize PyCaret regression setup ===
fd_exp = setup(
    data=train_df,                            # Training data from base currency pairs
    target='normalized_fd',                   # Target variable: Fractal Dimension
    session_id=42,                            # Seed for reproducibility
    verbose=False,                            # Suppress detailed logs
    ignore_features=['currency_pair', 'hour_start', 'normalized_volatility']  # Exclude irrelevant features
)

# === Compare candidate models using 2-fold CV ===
fd_model = compare_models(fold=2)

# === Hyperparameter tuning on the best model ===
fd_model_tuned = tune_model(fd_model, fold=2)

# === Predict on test set (remaining currency pairs) ===
# Reuses test_df_predict that excludes actual target columns
test_df['predicted_fd'] = predict_model(fd_model_tuned, data=test_df_predict)['prediction_label']


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ridge,Ridge Regression,0.0417,0.0021,0.0460,0.0351,0.0271,0.0602,0.0200
br,Bayesian Ridge,0.0407,0.0023,0.0477,-0.0402,0.0281,0.0589,0.0200
et,Extra Trees Regressor,0.0387,0.0023,0.0474,-0.0495,0.0280,0.0566,0.1300
llar,Lasso Least Angle Regression,0.0410,0.0023,0.0481,-0.0583,0.0283,0.0592,0.0150
lightgbm,Light Gradient Boosting Machine,0.0410,0.0023,0.0481,-0.0583,0.0283,0.0592,0.0850
dummy,Dummy Regressor,0.0410,0.0023,0.0481,-0.0583,0.0283,0.0592,0.0250
en,Elastic Net,0.0410,0.0023,0.0481,-0.0583,0.0283,0.0592,0.0150
lasso,Lasso Regression,0.0410,0.0023,0.0481,-0.0583,0.0283,0.0592,0.0200
rf,Random Forest Regressor,0.0415,0.0025,0.0503,-0.1684,0.0297,0.0600,0.1200
gbr,Gradient Boosting Regressor,0.0461,0.0026,0.0509,-0.2322,0.0300,0.0657,0.0550


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0422,0.0024,0.0492,-0.0059,0.0286,0.0591
1,0.0387,0.0020,0.0448,-0.0172,0.0268,0.0583
Mean,0.0404,0.0022,0.0470,-0.0115,0.0277,0.0587
Std,0.0018,0.0002,0.0022,0.0057,0.0009,0.0004


Fitting 2 folds for each of 10 candidates, totalling 20 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [16]:
# 8. Classification Phase: Label data based on mean values of volatility and FD

# === Combine training and testing sets to compute global averages ===
features_df = pd.concat([train_df, test_df], axis=0)

# === Compute global (across all data) means ===
mean_vol = features_df['normalized_volatility'].mean()
mean_fd = features_df['normalized_fd'].mean()

# === Define labeling logic ===
# - FORECASTABLE: both volatility and FD are below average
# - NON-FORECASTABLE: both are above average
# - PARTIALLY FORECASTABLE: one above, one below
def classify_predictability(row, vol_col, fd_col):
    if row[vol_col] < mean_vol and row[fd_col] < mean_fd:
        return 'FORECASTABLE'
    elif row[vol_col] > mean_vol and row[fd_col] > mean_fd:
        return 'NON-FORECASTABLE'
    else:
        return 'PARTIALLY FORECASTABLE'

# === Reload training/testing sets for classification (ensuring raw values) ===
train_df_clf, test_df_clf = split_train_test(df, base_cps, remaining_cps)

# === Apply labeling function to training set only (so far) ===
train_df_clf['class'] = train_df_clf.apply(
    lambda row: classify_predictability(row, 'normalized_volatility', 'normalized_fd'),
    axis=1
)

test_df_clf['class'] = test_df_clf.apply(
    lambda row: classify_predictability(row, 'normalized_volatility', 'normalized_fd'),
    axis=1
)


In [8]:
# 9. Output the classification results

print("📊 Train Classification Label Preview:")
print(train_df_clf[['currency_pair', 'normalized_volatility', 'normalized_fd', 'class']])


📊 Classification Label Preview:
   currency_pair  normalized_volatility  normalized_fd                   class
6         USDAUD               0.001074       0.716960        NON-FORECASTABLE
7         USDAUD               0.000865       0.709113        NON-FORECASTABLE
8         USDCAD               0.000332       0.655714            FORECASTABLE
9         USDCAD               0.000282       0.725877  PARTIALLY FORECASTABLE
10        USDCHF               0.000650       0.629937  PARTIALLY FORECASTABLE
11        USDCHF               0.000638       0.539399  PARTIALLY FORECASTABLE
14        USDEUR               0.000440       0.722500  PARTIALLY FORECASTABLE
15        USDEUR               0.000440       0.778728  PARTIALLY FORECASTABLE
16        USDGBP               0.000332       0.663569  PARTIALLY FORECASTABLE
17        USDGBP               0.000359       0.664684  PARTIALLY FORECASTABLE


In [18]:
print("📊 Test Classification Label Preview:")
print(test_df_clf[['currency_pair', 'normalized_volatility', 'normalized_fd', 'class']])


📊 Test Classification Label Preview:
   currency_pair  normalized_volatility  normalized_fd                   class
0         EURCHF               0.000331       0.722329  PARTIALLY FORECASTABLE
1         EURCHF               0.000363       0.537074            FORECASTABLE
2         GBPCHF               0.000517       0.759868  PARTIALLY FORECASTABLE
3         GBPCHF               0.000462       0.852012  PARTIALLY FORECASTABLE
4         GBPEUR               0.000365       0.715072  PARTIALLY FORECASTABLE
5         GBPEUR               0.000340       0.609898            FORECASTABLE
18        USDINR               0.000042       0.160000            FORECASTABLE
19        USDINR               0.001603       0.657912  PARTIALLY FORECASTABLE
20        USDJPY               0.000678       0.733894        NON-FORECASTABLE
21        USDJPY               0.000714       0.670798        NON-FORECASTABLE
